# C1 · PSF cromática (Moffat/Psfao)

**Spec:** [`docs/spec_C1_codex_chromatic_psf.md`](../docs/spec_C1_codex_chromatic_psf.md)  |  **Bloque:** C · Extracción  |  **Run de este set:** `ROXs42Bb_realigned`

Ajusta la PSF cromática de la primaria por bin de λ, comparando Moffat y Psfao y seleccionando la mejor.

| | |
|---|---|
| **Entrada** | Cubo alineado + posición primaria |
| **Salida (QC/productos)** | `stages/stage_e01_qc.json`, `stages/psf_model.json` |
| **Consume aguas abajo** | C3, C4, E4 (modelo de PSF) |


## Qué hace C1 y la decisión Moffat vs Psfao

C1 ajusta la **PSF cromática de la primaria** (por bin de λ) para modelar el halo estelar y poder tenerlo en cuenta en la posición del compañero. Ajusta **dos formas** por bin — un doble **Moffat** y un **Psfao** físico (`maoppy.Psfao`, PSF de óptica adaptiva NFM) — y **selecciona** por una métrica canónica: el residuo en el **anillo del compañero** (radio ≈ 71 px, ancho 3 px).

**Por qué Psfao:** el halo AO de NFM es ancho y un Moffat no lo ajusta bien en el rojo; Psfao es el modelo físico del halo AO.

**Decisión (blocker #8 cerrado):** selección automática por residuo mediano del anillo → **Psfao 4.44 % < Moffat 15.65 %** → se elige **Psfao**. El **híbrido** azimutal se probó y se **descartó** (con Psfao empeoraba: la escala venía de la FWHM inflada del Moffat).

**El nexo con B6/D1 (importante):** aun con Psfao, el **p90** del residuo del anillo es ~31 % (17 bins por encima del 5 %) — los bins del **rojo lejano** no cierran. Ese es el sistemático **B6** que reaparece en D1 como `divergent_continuum` y se acepta como presupuestado ([`docs/d1_canonical_method_decision.md`](../docs/d1_canonical_method_decision.md)). La PSF física **no** lo elimina — es el piso a esta geometría.

**Producto:** `psf_model.json` con los parámetros Psfao (`r0`, `beta`, …) por bin, suavizados con un polinomio; lo consumen C3/C4/E4.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_e01_psf.sh --run-id $RUN
```

Moderado (Psfao con lru_cache; minutos).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_e01_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_e01_psf.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_e01_qc.json', RUN_ID)
nb.show(qc, keys=['form_chosen', 'reason', 'ring_residual_pct_median', 'bins_above_5pct', 'n_ok_bins'], title='C1')


## Resultados que llevaron a la conclusión

Comparación de modelos, métrica del anillo y ajuste del `stage_e01_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('C1', 'stages/stage_e01_qc.json'):
        q = nb.load_qc('stages/stage_e01_qc.json', RUN_ID)
        mc = q['model_comparison']; rm = q['companion_ring_metric']; fit = q['fit']
        print('forma elegida:', mc['form_chosen'], '|', mc['reason'])
        print(f"  Moffat: mediana {mc['moffat']['ring_residual_pct_median']:.2f}%  p90 {mc['moffat']['ring_residual_pct_p90']:.1f}%  ({mc['moffat']['n_bins']} bins)")
        print(f"  Psfao : mediana {mc['psfao']['ring_residual_pct_median']:.2f}%  p90 {mc['psfao']['ring_residual_pct_p90']:.1f}%  ({mc['psfao']['n_bins']} bins)")
        print()
        print(f"anillo del compañero: radio {rm['radius_px']:.1f} px, ancho {rm['width_px']:.0f} px; "
              f"residuo mediano {rm['residual_pct_median']:.2f}% (p90 {rm['residual_pct_p90']:.1f}%), "
              f"bins >5% = {rm['bins_above_5pct']}  <-- B6 rojo lejano")
        print(f"ajuste: {fit['model']} (form={fit['form_chosen']}), fit_radius {fit['fit_radius_px']:.0f} px, "
              f"{fit['n_ok_bins']} bins OK / {fit['n_bins_rejected']} rechazados")
        print(f"híbrido aplicado: {q['hybrid']['applied']}")


## Plot 1 — la decisión: Moffat vs Psfao (residuo del anillo)

Del `stage_e01_qc.json` (barato). Psfao baja la **mediana** del residuo del anillo por debajo del objetivo 5 %; Moffat no. El **p90** de ambos sigue alto (~31–34 %) = el residuo B6 del rojo lejano que ni Psfao cierra.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_e01_qc.json', RUN_ID); mc = q['model_comparison']
    med = [mc['moffat']['ring_residual_pct_median'], mc['psfao']['ring_residual_pct_median']]
    p90 = [mc['moffat']['ring_residual_pct_p90'], mc['psfao']['ring_residual_pct_p90']]
    x = np.arange(2); cols = ['tab:red', 'tab:green']
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.bar(x - 0.18, med, 0.36, color=cols, label='mediana')
    ax.bar(x + 0.18, p90, 0.36, color=cols, alpha=0.45, label='p90')
    ax.axhline(5, color='k', ls='--', lw=1, label='objetivo <5%')
    for i, v in enumerate(med): ax.text(i - 0.18, v + 0.6, f'{v:.1f}%', ha='center', fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(['Moffat', 'Psfao'])
    ax.set_ylabel('residuo del anillo del compañero [%]')
    ax.set_title(f"C1 · selección: {mc['form_chosen'].upper()} (mediana {med[1]:.1f}% < {med[0]:.1f}%)")
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c1_psf'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'moffat_vs_psfao.png', dpi=110)
    print('figura ->', outdir / 'moffat_vs_psfao.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — la PSF cromática: r0 (Fried) vs λ

Del `psf_model.json` `param_table` (barato). El parámetro de Fried `r0` **sube con λ** (mejor seeing en el rojo, ~r0 ∝ λ^1.2) — por eso la PSF es *cromática* y se ajusta por bin. Es el modelo que C3/C4/E4 consumen.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    p = nb.load_qc('stages/psf_model.json', RUN_ID); pt = p['param_table']
    lam = np.array(pt['lambda_A']); r0 = np.array(pt['r0'], dtype=float)
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.plot(lam, r0, 'o-', color='tab:blue', ms=4)
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('r0 (parámetro de Fried) [m]')
    ax.set_title(f"C1 · PSF cromática {p['form'].upper()} ({p['system']}): r0 sube al rojo (mejor seeing)")
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c1_psf'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'chromatic_r0.png', dpi=110)
    print('figura ->', outdir / 'chromatic_r0.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Blocker #8 CERRADO**: C1 ajusta Moffat **y** Psfao por bin y selecciona por menor residuo del anillo (empate→Moffat). · [`docs/d1_canonical_method_decision.md`](../docs/d1_canonical_method_decision.md)
- **Forma elegida = psfao** (residuo de anillo mediano 4.44% vs 15.65% Moffat). `psf_model.json` byte-idéntico a la etapa lateral: consolidación neutra.
- Híbrido azimutal **descartado**: con Psfao empeoraba (la escala venía de la FWHM inflada del Moffat).
- **Residuo B6 del rojo lejano NO se cierra** (p90 ~31%, 17 bins >5%): es el piso a esta geometría → sistemática presupuestada que reaparece en D1 (`divergent_continuum`). · [`docs/d1_canonical_method_decision.md`](../docs/d1_canonical_method_decision.md)


## Conclusión (registrada)

**C1: PSF cromática ajustada; forma = Psfao (auto); residuo del anillo mediano 4.44% (<5% objetivo).**

- **Fecha:** consolidación WP-5, 2026-07-10 (commit `c3704b8`).
- **Ajuste:** `maoppy.Psfao` (muse_nfm), fit_radius 78 px, 43 bins OK / 3 rechazados; anillo del compañero radio 71 px.
- **Selección:** Psfao 4.44% < Moffat 15.65% (mediana del residuo del anillo); híbrido descartado.
- **B6 abierto/aceptado:** p90 ~31% (17 bins >5%) en el rojo lejano — la PSF física no lo cierra; sistemática presupuestada (D1 `divergent_continuum`).
- **Consolidación neutra:** `psf_model.json` byte-idéntico al de la etapa lateral previa.
- **Downstream:** `psf_model.json` (params Psfao por bin) lo consumen C3, C4 y E4.
